# A3.6 · Runtime containment levers

**Function A — Security Architecture & Platform → The Platform & Cloud Security Engineer**  ·  *Security of AI*

---

**Risk.** The stop lever is built during the incident.

**Control.** Throttle, scope-reduce, reroute, force HITL, hard stop — built and tested first.

**This lab.** Build the five containment levers before you need them.

| | |
|---|---|
| Open-source tooling | Falco, Kyverno, kagent |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A3.6"))

When containment fails, the runtime levers are what you have left. They are worth rehearsing before you need them.

In [ ]:
from cybercommons import ir, soc
import time

# the lever question: how much damage happens while containment waits?
for approval_minutes in (0.5, 5, 30):
    r = ir.containment_race(agent_actions_per_min=120,
                            human_approval_minutes=approval_minutes)
    print(f"human approval {approval_minutes:>4} min → "
          f"{r['actions_during_manual_approval']:>7.0f} actions "
          f"vs {r['actions_during_auto_containment']:>5.0f} automated "
          f"({r['ratio']}× more)")
print("\n", ir.containment_race(120, 5)["conclusion"])

Now the detection side: the levers only fire if something notices.

In [ ]:
events = [soc.Event(time.time(), "patch-agent", "http_get",
                    "http://169.254.169.254/latest/meta-data/iam/")]
for a in soc.run_rules(events, soc.default_rules()):
    print(f"[{a.severity}] {a.rule}\n    → {a.response}")

### Expect

A five-minute human approval permits ~600 agent actions against ~10 for automated containment — a 60× difference. The metadata rule fires at critical severity with a concrete response.

### Your turn

Rank your available levers by measured time-to-effect: revoke identity, kill process, network quarantine, rotate credentials. The fastest one should be the one you have automated.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A3.6.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*